# BearingRangeFactor

A `BearingRangeFactor` combines a direction and distance observation in one binary factor for landmark or relative-pose estimation.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sam/doc/BearingRangeFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

from gtsam.symbol_shorthand import L, X

## Create a bearing-range factor

Start with a pose and landmark, then compute the direction and distance that the pose would observe. `BearingRangeFactor2D` stores both values in one measurement. Its two-dimensional noise model is ordered as bearing uncertainty followed by range uncertainty.

In [3]:
pose_key, landmark_key = X(0), L(0)
pose = gtsam.Pose2(1.0, 2.0, 0.3)
landmark = np.array([4.0, 5.0])
bearing = pose.bearing(landmark)
distance = pose.range(landmark)
measurement_noise = gtsam.noiseModel.Diagonal.Sigmas(
    np.array([0.01, 0.1])
)

factor = gtsam.BearingRangeFactor2D(
    pose_key, landmark_key, bearing, distance, measurement_noise
)
values = gtsam.Values()
values.insert(pose_key, pose)
values.insert(landmark_key, landmark)

measurement = factor.measured()
assert np.allclose(factor.unwhitenedError(values), 0.0)
print("bearing:", measurement.bearing().theta())
print("range:", measurement.range())

bearing: 0.4853981633974484
range: 4.242640687119285


## Estimate a landmark

With the pose fixed by a prior, one bearing-range observation determines the landmark. We add the factor to a small nonlinear graph, perturb the landmark's initial value, and let Levenberg–Marquardt recover the measured location.

In [4]:
graph = gtsam.NonlinearFactorGraph()
graph.addPriorPose2(
    pose_key,
    pose,
    gtsam.noiseModel.Diagonal.Sigmas(np.array([1e-4, 1e-4, 1e-4])),
)
graph.add(factor)

initial = gtsam.Values()
initial.insert(pose_key, pose)
initial.insert(landmark_key, landmark + np.array([0.4, -0.2]))
result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()

assert np.allclose(result.atPoint2(landmark_key), landmark, atol=1e-6)
print("estimated landmark:", result.atPoint2(landmark_key))

estimated landmark: [4. 5.]


## When to use it

Use a `BearingRangeFactor` when one sensor observation supplies both direction and distance. `BearingRangeFactor3D` uses a `Unit3` bearing for spatial landmarks; the pose-to-pose variants use the same constructor and optimization workflow.

The separate [`BearingFactor`](BearingFactor.ipynb) and [`RangeFactor`](RangeFactor.ipynb) pages show how to model the two measurements independently.

## Source

[`BearingRangeFactor.h`](../BearingRangeFactor.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.